# Governed Invoice Quality Workflow

A synthetic, credential-free example of bronze → silver → gold data processing, quality checks, privacy redaction, and human-reviewed incident triage. Run this notebook on Kaggle with Internet off.


In [ ]:
import csv, hashlib, json, re
from collections import defaultdict
from datetime import datetime
from pathlib import Path


## 1. Bronze: synthetic source records
No personal or company data is included.


In [ ]:
csv_text = '''invoice_id,vendor_id,invoice_date,amount,currency,description,owner_email
INV-001,V-10,2026-01-01,1250.00,USD,Office supplies contact alice@example.test,alice@example.test
INV-002,V-20,2026-01-02,750.50,USD,Subscription,bob@example.test
INV-002,V-20,2026-01-03,80.00,USD,Duplicate,bob@example.test
INV-004,V-10,2026-99-02,-30,USD,Correction,alice@example.test
INV-005,V-30,2026-01-05,450.00,USD,Maintenance,charlie@example.test
'''
from io import StringIO
rows = list(csv.DictReader(StringIO(csv_text)))
print('Bronze rows:', len(rows))


## 2. Silver and gold: validate, redact, aggregate


In [ ]:
required = ('invoice_id','vendor_id','invoice_date','amount','currency','description','owner_email')
seen, clean, quarantine = set(), [], []
for row in rows:
    errors = []
    if any(not row.get(k,'').strip() for k in required): errors.append('missing_required_field')
    if row['invoice_id'] in seen: errors.append('duplicate_invoice_id')
    seen.add(row['invoice_id'])
    try:
        amount = float(row['amount'])
        if amount <= 0: errors.append('nonpositive_amount')
    except ValueError: errors.append('invalid_amount'); amount = 0
    try: datetime.strptime(row['invoice_date'],'%Y-%m-%d')
    except ValueError: errors.append('invalid_invoice_date')
    if errors: quarantine.append({'invoice_id':row['invoice_id'],'reasons':errors})
    else: clean.append({'invoice_id':row['invoice_id'],'vendor_id':row['vendor_id'], 'amount':amount, 'description':re.sub(r'\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b','[EMAIL_REDACTED]',row['description'])})
totals = defaultdict(float)
for row in clean: totals[row['vendor_id']] += row['amount']
summary = {'input_rows':len(rows),'accepted_rows':len(clean),'rejected_rows':len(quarantine),'vendor_totals':dict(totals)}
print(json.dumps(summary,indent=2)); print('Quarantine:', quarantine)


## 3. Governed triage
The template below illustrates prompt engineering; no model is called. Send only aggregated counts, require JSON output, reject unsupported claims, and have a person approve actions.


In [ ]:
prompt_template = '''You are a data-quality triage assistant. Use only the supplied aggregate metrics and error counts. Return JSON with severity, top_issue, recommended_action, human_review_required. Do not invent root causes or expose source records. No automated data deletion. Metrics: {metrics}'''
counts = {reason:sum(reason in item['reasons'] for item in quarantine) for item in quarantine for reason in item['reasons']}
prompt = prompt_template.format(metrics=json.dumps({'summary':summary,'error_counts':counts}))
triage = {'severity':'high' if len(clean)/len(rows)<0.8 else 'low','top_issue':max(counts,key=counts.get) if counts else 'none','human_review_required':bool(counts),'recommended_action':'Inspect source mapping and replay corrected quarantined records.'}
print(json.dumps(triage,indent=2)); print('Source SHA256:',hashlib.sha256(csv_text.encode()).hexdigest())


## 4. Extension
In production: schedule an n8n workflow, run the Python job on a managed service, write governed Delta tables to Azure Databricks/ADLS, and route only an approved summary to the notification channel. Provisioning, authentication, access control, and retention policies are deployment tasks.
